In [2]:
import os
import sys
import session_info
import geopandas as gpd
import numpy as np
import scipy
import pandas as pd
pd.set_option('future.no_silent_downcasting', True)
import matplotlib.pyplot as plt
import mpl_toolkits
import rtree
from shapely.geometry import Polygon

"""
Reads  Counties 2023 census data
Only uses lower 48 states
Updates percison and projection
Calculates Polsby Popper for each county


"""

Shapepath = "Census data/"
Dataoutpath ="Process data/"

Projection = "ESRI:102008"
#session_info.show()
print( "Libraries loaded")

Libraries loaded


In [3]:
def Border_clip( Geodf, Borderdf):
    """
    Takes a GeoPandas dataframe and clips is borders to the Borders in Borderdf
    Assumes GeoPandas dataframe has rows delineated by GEOID
    Cleans up any non polygon residuals
    Removes any remaining with less than 100,000 sq meters in area
    """
    Districts_clipped = gpd.clip( Geodf, Borderdf, keep_geom_type=False).copy()
    Allgeoms = Districts_clipped.explode()
    Allpolys = Allgeoms[ Allgeoms.geometry.geom_type=="Polygon"].copy()
    Allpolys['Area'] = Allpolys.area
    Allpolys = Allpolys[ Allpolys['Area'] >100000]
    DistrictsM = Allpolys.dissolve(by='GEOID', method='coverage').reset_index()
    DistrictsM.drop(['Area'], axis=1, inplace=True)
    return( DistrictsM)

print( "Border_clip function defined")
    

Border_clip function defined


In [4]:
def Geo_check( Geodf ) :
    """
    Analyze the composition of the geo data frame
    Looks for differing geometry types
    """
    ### Count types of geometries found pre clipping
    geometry_types = {'Point': 0, 'LineString': 0, 'Polygon': 0, 'MultiPolygon': 0, 'GeometryCollection':0}

    for geom in Geodf['geometry']:
        geometry_types[geom.geom_type] += 1 

    print("Points:", geometry_types['Point'])
    print("Lines:", geometry_types['LineString'])
    print("Polygons:", geometry_types['Polygon'])
    print("MultiPolygons:", geometry_types['MultiPolygon'])
    print("GeometryCollections:", geometry_types['GeometryCollection'])
    print("Total Polys:", len( Geodf.explode()))
    print( "Total rows:", len(Geodf) )
    print( "Min area polygon:", int( min( Geodf.explode().area)))
    return
    
print( "Geo_check function defined")

Geo_check function defined


In [5]:
def Polsby(Geometry):
    """
    Calculates the Polsby Poppper compactness ratio of a given geometry. 
    Uses an area weighted average if more than one polygon.
    Uses only the exterior ring of each polgon
    Assumes coordinates have already been projected     
    Args: 
        Geometry (shapely geometry): A Shapely geometry object (e.g., Polygon, MultiPolygon). 
        
    Returns:
        float: The PolsbyPopper score
    """
    Geos = gpd.GeoDataFrame( geometry=[Geometry], crs=Projection)
    #Darea = Geos["geometry"].area
    Geos = Geos.explode( )
    Geos['geometry'] = Geos['geometry'].apply(lambda p: Polygon(p.exterior.coords)) ## make polygon out of exterior corrdinates
    Geos['Perimeter2'] = Geos["geometry"].length**2
    Geos["Area"] = Geos["geometry"].area
    Darea = Geos['Area'].sum()
    Geos["PolsbyI"] = (Geos["Area"] * 4 * 3.14159) /  Geos["Perimeter2"] 
    Geos["PolsbyW"] = Geos["PolsbyI"] * Geos["Area"] / Darea
    Polsby = Geos["PolsbyW"].sum() 
    
    return Polsby
print( "Function defined")   

Function defined


In [6]:
##### Load US boundary file for clipping
File = 'cb_2023_us_nation_20m'
Shapedatafile = Shapepath + File + ".shp"
RawShapes = gpd.read_file( Shapedatafile)
USbox = Polygon( [ (-130,20),(-60,20), (-60,50), (-130,50) ] )        ### Contental US box
Shapes = gpd.clip( RawShapes, USbox)
#Shapes.drop( ['REGIONCE','NAMELSAD', 'LSAD', 'ALAND','AWATER'] , axis=1, inplace=True)
Shapes.to_crs( Projection, inplace=True)
Shapes.geometry = Shapes.geometry.set_precision( 0.01 )
#Shapes.plot()
print( "Loaded ", File," with ", len( Shapes.explode()), " polygons")
#Shapes.head()
US48 = Shapes.copy()

Loaded  cb_2023_us_nation_20m  with  23  polygons


In [10]:
File = 'tl_2023_us_county'
Shapedatafile = Shapepath + File + ".shp"
RawCounties = gpd.read_file( Shapedatafile)
RawCounties.drop( ['NAMELSAD','GEOIDFQ', 'LSAD','CLASSFP', 'MTFCC', 'CSAFP', 'CBSAFP', 'METDIVFP', 'FUNCSTAT', 'ALAND','AWATER', 'INTPTLAT', 'INTPTLON'] , axis=1, inplace=True)
USbox = Polygon( [ (-130,20),(-60,20), (-60,50), (-130,50) ] )        ### Contental US box
Counties = gpd.clip( RawCounties, USbox)
Counties.to_crs( Projection, inplace=True)
Counties.geometry = Counties.geometry.set_precision(0.01)
### Rename GEOIDs that are independent city with dupe county name
Change_name_geoid = ["51620","51770","51760","51600","29510","24510"]  
for geoid in Change_name_geoid:
    Counties.loc[Counties['GEOID'] == geoid, 'NAME'] += '_City'  ### appends _city to each county name

print(  File +" loaded ", len(Counties)," rows of data")
#Counties.plot()
#Counties.head()
#Counties.tail()


tl_2023_us_county loaded  3109  rows of data


,STATEFP,COUNTYFP,COUNTYNS,GEOID,NAME,geometry
2772,48,043,01383807,48043,Brewster,"POLYGON ((-636769.01 -1209452.4, -636822.25 -1..."
1236,48,377,01383974,48377,Presidio,"POLYGON ((-824486.62 -1057370.66, -824465.7 -1..."
2141,48,443,01384007,48443,Terrell,"POLYGON ((-607841.95 -1140783.7, -607811.08 -1..."
2512,48,105,01383838,48105,Crockett,"POLYGON ((-583537.16 -1023537.35, -583528.62 -..."
1330,48,371,01383971,48371,Pecos,"POLYGON ((-570835.69 -1080968.5, -570877.43 -1..."


In [20]:
### Clip borders to overall US
Districts = Border_clip( Counties, US48 )
print( 'Number counties post clipping', len(Districts))
#Geo_check( Districts )
print('done clipping')

done clipping


In [23]:
#### Calc county metrics metrics
Districts['PolsbyW'] = Districts.geometry.apply( Polsby )
print( 'Done with county level metrics')
#Districts.head()

Done with county level metrics


,GEOID,geometry,STATEFP,COUNTYFP,COUNTYNS,NAME,PolsbyW
0,01001,"POLYGON ((822188.34 -848112.17, 822177.79 -848...",01,001,00161526,Autauga,0.441517
1,01003,"POLYGON ((739461.79 -997749.71, 739467.05 -997...",01,003,00161527,Baldwin,0.305894
2,01005,"POLYGON ((947483.22 -929127.09, 947235.85 -929...",01,005,00161528,Barbour,0.401024
3,01007,"POLYGON ((791177.08 -754987.58, 791414.54 -754...",01,007,00161529,Bibb,0.559660
4,01009,"POLYGON ((827587.45 -636860.06, 827596.26 -636...",01,009,00161530,Blount,0.361476


In [24]:
Column_order = [ 'GEOID', 'STATEFP', 'COUNTYFP', 'COUNTYNS','NAME', 'PolsbyW', 'geometry' ]
Districts = Districts.reindex(columns=Column_order)
Shapeoutfile = Dataoutpath + 'County2023.shp'
Districts.to_file( Shapeoutfile )
print( 'Done writing ', len( Districts ), 'rows to ', Shapeoutfile)
#Districts.plot()
Geo_check( Districts )
CountyData = pd.DataFrame(Districts.drop(columns='geometry'))
CSVoutfile = Dataoutpath + 'County2023.csv'
CountyData.to_csv( CSVoutfile)
#Districts.head()

Done writing  3109 rows to  Process data/County2023.shp
Points: 0
Lines: 0
Polygons: 3055
MultiPolygons: 54
GeometryCollections: 0
Total Polys: 3198
Total rows: 3109
Min area polygon: 111949


,GEOID,STATEFP,COUNTYFP,COUNTYNS,NAME,PolsbyW,geometry
0,01001,01,001,00161526,Autauga,0.441517,"POLYGON ((822188.34 -848112.17, 822177.79 -848..."
1,01003,01,003,00161527,Baldwin,0.305894,"POLYGON ((739461.79 -997749.71, 739467.05 -997..."
2,01005,01,005,00161528,Barbour,0.401024,"POLYGON ((947483.22 -929127.09, 947235.85 -929..."
3,01007,01,007,00161529,Bibb,0.559660,"POLYGON ((791177.08 -754987.58, 791414.54 -754..."
4,01009,01,009,00161530,Blount,0.361476,"POLYGON ((827587.45 -636860.06, 827596.26 -636..."
